# Módulo 05 · Combinar datos: `merge`, validaciones, `concat` y `merge_asof`

**Curso de pandas intermedio → avanzado** · Dataset: *Olist* (muestras en `data/`)

## Objetivos
1. Razonar sobre la **cardinalidad** de las llaves (1:1, 1:m, m:m) antes de unir.
2. Dominar `merge`: `how`, llaves con distinto nombre, sufijos, `indicator` y **`validate`**.
3. Detectar y evitar la **explosión de filas** (el error que infla totales en silencio).
4. Hacer *semi-joins* y *anti-joins* ("los que sí están" / "los que no están").
5. Unir por índice con `join` y apilar con `concat`.
6. Hacer uniones **por cercanía en el tiempo** con `merge_asof`.
7. Completar datos de dos fuentes con `combine_first`.

| Sección | Tema |
|---|---|
| 0 | Setup y carga |
| 1 | Cardinalidad de llaves |
| 2 | `merge` a fondo |
| 3 | `validate`: contratos sobre tus datos |
| 4 | La explosión de filas |
| 5 | Semi-joins y anti-joins |
| 6 | Caso práctico: geolocalización |
| 7 | `join` y uniones por índice |
| 8 | `concat` |
| 9 | `merge_asof` |
| 10 | `combine_first` |
| ✍️ | Ejercicios · ✅ Soluciones |

## 0 · Setup y carga

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)


def pendiente(*valores) -> bool:
  # True si algún valor sigue siendo `...` (ejercicio sin resolver)
  return any(v is ... for v in valores)


print("pandas", pd.__version__)

In [ ]:
orders = pd.read_csv(
  DATA / "olist_orders_dataset.csv",
  dtype={"order_status": "category"},
  parse_dates=["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
)
items = pd.read_csv(
  DATA / "olist_order_items_dataset.csv",
  dtype={"order_item_id": "int8"},
  parse_dates=["shipping_limit_date"],
)
payments = pd.read_csv(
  DATA / "olist_order_payments_dataset.csv",
  dtype={"payment_type": "category", "payment_sequential": "int8", "payment_installments": "int8"},
)
reviews = pd.read_csv(
  DATA / "olist_order_reviews_dataset.csv",
  usecols=["review_id", "order_id", "review_score", "review_creation_date", "review_answer_timestamp"],
  parse_dates=["review_creation_date", "review_answer_timestamp"],
)
customers = pd.read_csv(
  DATA / "olist_customers_dataset.csv",
  dtype={"customer_zip_code_prefix": "str", "customer_state": "category"},
).assign(customer_zip_code_prefix=lambda d: d["customer_zip_code_prefix"].str.zfill(5))

sellers = pd.read_csv(
  DATA / "olist_sellers_dataset.csv",
  dtype={"seller_zip_code_prefix": "str", "seller_state": "category"},
).assign(seller_zip_code_prefix=lambda d: d["seller_zip_code_prefix"].str.zfill(5))

geo = pd.read_csv(
  DATA / "olist_geolocation_dataset.csv",
  dtype={"geolocation_zip_code_prefix": "str", "geolocation_state": "category"},
).assign(geolocation_zip_code_prefix=lambda d: d["geolocation_zip_code_prefix"].str.zfill(5))

products = pd.read_csv(DATA / "olist_products_dataset.csv",
                       usecols=["product_id", "product_category_name"])

## 1 · Cardinalidad de llaves

Antes de cualquier `merge`, pregúntate **cuántas filas tiene cada valor de la llave en cada lado**:

| Relación | Ejemplo en Olist | Filas del resultado |
|---|---|---|
| 1:1 | `orders` ↔ `customers` (por `customer_id`) | ≤ filas del lado izquierdo |
| 1:m | `orders` → `items` | = filas del lado "muchos" |
| m:1 | `items` → `products` | = filas del lado "muchos" |
| **m:m** | `items` ↔ `payments` (por `order_id`) | ⚠️ **producto cartesiano por llave** |

Una función de diagnóstico rápido:

In [ ]:
def perfil_llave(df: pd.DataFrame, llave) -> pd.Series:
  conteo = df.groupby(llave).size()
  return pd.Series({
    "filas": len(df),
    "llaves_unicas": len(conteo),
    "max_repeticiones": conteo.max(),
    "llaves_repetidas": (conteo > 1).sum(),
    "nulos_en_llave": df[llave].isna().any(axis=None) if isinstance(llave, list) else df[llave].isna().sum(),
  })


pd.DataFrame({
  "orders.order_id": perfil_llave(orders, "order_id"),
  "items.order_id": perfil_llave(items, "order_id"),
  "payments.order_id": perfil_llave(payments, "order_id"),
  "reviews.order_id": perfil_llave(reviews, "order_id"),
  "geo.zip": perfil_llave(geo, "geolocation_zip_code_prefix"),
})

Hallazgos que condicionan todo el módulo:

- `orders.order_id` es único ✅.
- `items`, `payments` **y `reviews`** repiten `order_id` (¡hay pedidos con dos reseñas!).
- `geo` repite cada prefijo postal **muchas veces**: unirla tal cual multiplicaría filas.

## 2 · `merge` a fondo

### Tipos de unión (`how`)

| `how` | Conserva |
|---|---|
| `"inner"` (default) | sólo llaves presentes en **ambos** |
| `"left"` | todas las de la izquierda |
| `"right"` | todas las de la derecha |
| `"outer"` | todas las de ambos |
| `"cross"` | producto cartesiano (sin llave) |

In [ ]:
for how in ["inner", "left", "right", "outer"]:
  r = orders.merge(reviews, on="order_id", how=how)
  print(f"{how:6}: {len(r)} filas")

### Llaves con distinto nombre y sufijos

Cuando la llave se llama distinto en cada lado: `left_on` / `right_on`. Cuando hay columnas repetidas que **no** son llave, pandas agrega sufijos (`_x`, `_y` por defecto); mejor elegirlos tú.

In [ ]:
zip_cliente = (
  orders[["order_id", "customer_id"]]
  .merge(customers, on="customer_id", how="left", validate="1:1")
)
zip_cliente.head(3)

In [ ]:
geo_zip = geo.groupby("geolocation_zip_code_prefix", as_index=False).agg(
  lat=("geolocation_lat", "median"),
  lng=("geolocation_lng", "median"),
  state=("geolocation_state", "first"),
)

zip_cliente.merge(geo_zip, left_on="customer_zip_code_prefix",
                  right_on="geolocation_zip_code_prefix", how="left").head(3)

Observa que `left_on`/`right_on` deja **ambas** columnas llave en el resultado. Normalmente eliminas una con `.drop(columns=...)`.

### `indicator=True`: ¿de dónde vino cada fila?

In [ ]:
m = orders[["order_id"]].merge(reviews[["order_id", "review_id"]], on="order_id",
                               how="outer", indicator=True)
m["_merge"].value_counts()

### Tipos de llave incompatibles

pandas **no** convierte tipos de llave por ti. Si un lado tiene el código postal como `int` y el otro como `str`, el `merge` falla (y si ambos fueran `str` pero uno con ceros y otro sin ellos, ¡no fallaría y simplemente no habría coincidencias!).

In [ ]:
customers_int = pd.read_csv(DATA / "olist_customers_dataset.csv")      # zip como int64
try:
  customers_int.merge(geo_zip, left_on="customer_zip_code_prefix", right_on="geolocation_zip_code_prefix")
except ValueError as e:
  print("ValueError:", str(e)[:90], "...")

In [ ]:
# El caso silencioso: ambos son str, pero uno sin zfill
sin_ceros = customers_int.assign(customer_zip_code_prefix=customers_int["customer_zip_code_prefix"].astype("str"))
con_ceros = customers

for nombre, df in [("sin zfill", sin_ceros), ("con zfill", con_ceros)]:
  r = df.merge(geo_zip, left_on="customer_zip_code_prefix",
               right_on="geolocation_zip_code_prefix", how="left")
  print(f"{nombre}: clientes con coordenadas = {r['lat'].notna().sum()} de {len(r)}")

## 3 · `validate`: contratos sobre tus datos

`validate` hace que `merge` **verifique** la cardinalidad esperada y lance `MergeError` si no se cumple. Es la forma más barata de evitar bugs silenciosos.

| `validate` | Exige llave única en… |
|---|---|
| `"1:1"` / `"one_to_one"` | ambos lados |
| `"1:m"` / `"one_to_many"` | lado izquierdo |
| `"m:1"` / `"many_to_one"` | lado derecho |
| `"m:m"` | nada (sólo documenta) |

In [ ]:
from pandas.errors import MergeError

try:
  orders.merge(reviews, on="order_id", how="left", validate="1:1")
except MergeError as e:
  print("MergeError:", str(e).splitlines()[0])

In [ ]:
# Los pedidos con más de una reseña
reviews[reviews["order_id"].duplicated(keep=False)].sort_values(["order_id", "review_answer_timestamp"])

Decidir **cuál** reseña conservar es una decisión de negocio. Por ejemplo, la más reciente:

In [ ]:
ultima_review = (
  reviews.sort_values("review_answer_timestamp")
         .drop_duplicates("order_id", keep="last")
)
orders_rev = orders.merge(ultima_review, on="order_id", how="left", validate="1:1")
print(len(orders), "->", len(orders_rev))

## 4 · La explosión de filas

Queremos un resumen por pedido con el total de ítems y el total pagado. El enfoque ingenuo une todo y luego suma:

In [ ]:
ingenuo = items.merge(payments, on="order_id")          # m:m por order_id
print("items:", len(items), "| payments:", len(payments), "| unión:", len(ingenuo))
print(f"Pagado real            : {payments['payment_value'].sum():>10,.2f}")
print(f"Pagado tras la unión   : {ingenuo['payment_value'].sum():>10,.2f}   <- ¡inflado!")

Un pedido con 2 ítems y 2 pagos genera 2 × 2 = 4 filas: cada pago aparece dos veces. No hay error, sólo un número equivocado.

✅ **Regla:** agrega cada tabla "muchos" **al nivel de la llave** antes de unir. Así cada `merge` es 1:1 o m:1.

In [ ]:
items_ped = items.groupby("order_id").agg(
  n_items=("order_item_id", "size"),
  precio=("price", "sum"),
  flete=("freight_value", "sum"),
)
pagos_ped = payments.groupby("order_id").agg(
  n_pagos=("payment_sequential", "size"),
  pagado=("payment_value", "sum"),
)

resumen = (
  orders[["order_id", "order_status"]]
  .merge(items_ped, on="order_id", how="left", validate="1:1")
  .merge(pagos_ped, on="order_id", how="left", validate="1:1")
)
print(f"Pagado tras la unión correcta: {resumen['pagado'].sum():,.2f}")
resumen.head()

> 🧪 **Test rápido de explosión:** después de cada `merge`, compara `len()` contra lo esperado, o usa `validate`. Si el número de filas creció y no lo esperabas, tienes una relación m:m escondida.

## 5 · Semi-joins y anti-joins

No siempre quieres **traer columnas**; a veces sólo quieres **filtrar** por existencia en otra tabla.

| Operación | Pregunta | Herramienta más simple |
|---|---|---|
| Semi-join | ¿qué filas de A **tienen** pareja en B? | `A[A["k"].isin(B["k"])]` |
| Anti-join | ¿qué filas de A **no tienen** pareja en B? | `A[~A["k"].isin(B["k"])]` |
| Anti-join con llave compuesta | ídem, con varias columnas | `merge(..., indicator=True)` + `query("_merge == 'left_only'")` |

In [ ]:
sin_review = orders[~orders["order_id"].isin(reviews["order_id"])]
sin_review[["order_id", "order_status", "order_purchase_timestamp"]]

In [ ]:
# Clientes cuyo prefijo postal no aparece en la tabla de geolocalización
anti = (
  customers.merge(geo_zip[["geolocation_zip_code_prefix"]],
                  left_on="customer_zip_code_prefix", right_on="geolocation_zip_code_prefix",
                  how="left", indicator=True)
  .query("_merge == 'left_only'")
  .drop(columns=["geolocation_zip_code_prefix", "_merge"])
)
print(f"Clientes sin coordenadas: {len(anti)} de {len(customers)}")
anti.head(3)

> ⚠️ Un semi-join con `merge(how="inner")` **duplica** filas si la llave se repite en B. Con `isin` nunca pasa.

## 6 · Caso práctico: geolocalización y distancias

¿A qué distancia viaja cada ítem del vendedor al cliente? Necesitamos coordenadas de ambos lados. La tabla `geo` tiene **muchas filas por prefijo**, así que ya la redujimos a una fila por prefijo (`geo_zip`, mediana de lat/lng) — la regla de la sección 4.

Primero, **cobertura**: ¿cuántos prefijos de cada lado encuentran coordenadas?

In [ ]:
cobertura = pd.Series({
  "clientes": customers["customer_zip_code_prefix"].isin(geo_zip["geolocation_zip_code_prefix"]).mean(),
  "vendedores": sellers["seller_zip_code_prefix"].isin(geo_zip["geolocation_zip_code_prefix"]).mean(),
}).mul(100).round(1)
cobertura

La muestra de geolocalización cubre bien a los clientes pero casi nada a los vendedores. Es una situación muy real: **una fuente de referencia incompleta**. Estrategia típica: usar el dato exacto cuando exista y un **respaldo** cuando no. Como respaldo usaremos las coordenadas aproximadas de la **capital** del estado del vendedor (tabla escrita a mano):

In [ ]:
capitales = pd.DataFrame(
  {"lat": [-23.55, -25.43, -19.92, -27.60, -22.91, -30.03, -16.69, -15.79, -15.60, -20.32, -8.05, -12.97],
   "lng": [-46.63, -49.27, -43.94, -48.55, -43.17, -51.23, -49.26, -47.88, -56.10, -40.34, -34.88, -38.50]},
  index=pd.Index(["SP", "PR", "MG", "SC", "RJ", "RS", "GO", "DF", "MT", "ES", "PE", "BA"], name="estado"),
)

coords = geo_zip.set_index("geolocation_zip_code_prefix")[["lat", "lng"]]

coord_vendedor = (
  sellers.set_index("seller_id")
  .join(coords, on="seller_zip_code_prefix")                       # exacto (si existe)
  [["lat", "lng"]]
  .combine_first(                                                  # respaldo: capital (sección 10)
    sellers.set_index("seller_id")[["seller_state"]].astype(str)
    .join(capitales, on="seller_state")[["lat", "lng"]]
  )
)
print("Vendedores sin coordenadas:", coord_vendedor["lat"].isna().sum())

Fórmula de *haversine* (distancia sobre la esfera), vectorizada con NumPy:

In [ ]:
def haversine_km(lat1, lng1, lat2, lng2):
  lat1, lng1, lat2, lng2 = map(np.radians, (lat1, lng1, lat2, lng2))
  a = (np.sin((lat2 - lat1) / 2) ** 2
       + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2)
  return 6371 * 2 * np.arcsin(np.sqrt(a))


# Prueba: São Paulo -> Rio de Janeiro ≈ 360 km
round(haversine_km(-23.55, -46.63, -22.91, -43.17))

In [ ]:
viajes = (
  items[["order_id", "order_item_id", "seller_id", "freight_value"]]
  .merge(orders[["order_id", "customer_id"]], on="order_id", how="left", validate="m:1")
  .merge(customers[["customer_id", "customer_zip_code_prefix"]], on="customer_id", how="left", validate="m:1")
  .join(coord_vendedor.add_prefix("v_"), on="seller_id")             # join por índice (sección 7)
  .join(coords.add_prefix("c_"), on="customer_zip_code_prefix")
)
viajes["km"] = haversine_km(viajes["v_lat"], viajes["v_lng"], viajes["c_lat"], viajes["c_lng"])

print("Filas:", len(items), "->", len(viajes))
viajes[["km", "freight_value"]].describe().round(1)

In [ ]:
ax = viajes.plot.scatter(x="km", y="freight_value", alpha=0.5, figsize=(7, 4),
                         title="Flete vs distancia vendedor → cliente")
ax.set_xlabel("km"); ax.set_ylabel("flete (BRL)");

> 💡 Con el dataset completo de Olist la cobertura de vendedores es casi total y el respaldo apenas se usa, pero el patrón **exacto → respaldo** con `combine_first` es el mismo.

## 7 · `join` y uniones por índice

`df.join(otro)` une **por el índice de `otro`**. Por defecto es `how="left"` y usa el índice de `df`, o la columna indicada en `on=`. Es cómodo cuando la tabla de referencia ya está indexada por su llave (como `coords` arriba).

| Quiero unir… | Usa |
|---|---|
| columna ↔ columna | `merge(on=...)` |
| columna ↔ índice | `df.join(ref, on="col")` o `merge(left_on="col", right_index=True)` |
| índice ↔ índice | `df.join(ref)` |
| traer **una** columna por llave única | `df["col"].map(ref_series)` (Módulo 02) |
| varios DataFrames por índice | `df.join([a, b, c])` |

In [ ]:
por_pedido = items_ped.join([pagos_ped, ultima_review.set_index("order_id")[["review_score"]]], how="left")
por_pedido.head()

## 8 · `concat`

`concat` **apila** objetos, sin buscar llaves:

- `axis=0` (default): una tabla debajo de otra. Las columnas se alinean **por nombre**; las que faltan quedan `NaN`.
- `axis=1`: lado a lado, alineando **por índice**.
- `keys=[...]`: agrega un nivel de índice que identifica el origen.

In [ ]:
# Eventos de cada pedido en formato largo: una tabla por tipo de evento
eventos = pd.concat(
  {
    "compra": orders[["order_id", "order_purchase_timestamp"]].rename(columns={"order_purchase_timestamp": "ts"}),
    "aprobado": orders[["order_id", "order_approved_at"]].rename(columns={"order_approved_at": "ts"}),
    "entregado": orders[["order_id", "order_delivered_customer_date"]].rename(columns={"order_delivered_customer_date": "ts"}),
  },
  names=["evento", None],
).reset_index(level="evento").dropna(subset=["ts"]).reset_index(drop=True)

eventos.sort_values(["order_id", "ts"]).head(6)

In [ ]:
# Columnas que no coinciden -> NaN (join="outer" por defecto) o sólo comunes (join="inner")
a = pd.DataFrame({"x": [1, 2], "y": [3, 4]})
b = pd.DataFrame({"x": [5], "z": [6]})
display(pd.concat([a, b], ignore_index=True))
display(pd.concat([a, b], ignore_index=True, join="inner"))

### Antipatrón: `concat` dentro de un bucle

Cada `concat` copia **todo** lo acumulado: el costo crece cuadráticamente. Acumula en una lista y concatena **una sola vez**.

In [ ]:
# ❌
resultado = pd.DataFrame()
for estado, grupo in customers.groupby("customer_state", observed=True):
  resultado = pd.concat([resultado, grupo.head(1)])

# ✅
partes = [grupo.head(1) for _, grupo in customers.groupby("customer_state", observed=True)]
resultado_bien = pd.concat(partes)

# ✅✅ (y sin bucle: Módulo 03)
resultado_mejor = customers.groupby("customer_state", observed=True).head(1)
len(resultado), len(resultado_bien), len(resultado_mejor)

## 9 · `merge_asof`

Une cada fila con la fila **más cercana en el tiempo** (u otra columna ordenada) de la otra tabla, en lugar de exigir igualdad exacta. Casos típicos: tipo de cambio vigente, precio vigente, lectura de sensor más reciente.

| Parámetro | Para qué |
|---|---|
| `on` / `left_on` / `right_on` | columna ordenada (fechas, números) |
| `by` | además, coincidir exactamente en esta llave (p. ej. por estado) |
| `direction` | `"backward"` (≤, default), `"forward"` (≥), `"nearest"` |
| `tolerance` | distancia máxima aceptada (si no, `NaN`) |
| `allow_exact_matches` | si `False`, exige estrictamente < o > |

**Requisito:** ambas tablas **ordenadas** por la columna `on`.

Ejemplo: una tabla (inventada para el curso) con la **comisión vigente** del marketplace por estado del vendedor, que cambia en ciertas fechas:

In [ ]:
comisiones = pd.DataFrame({
  "vigente_desde": pd.to_datetime(["2017-01-01", "2017-01-01", "2017-09-01", "2018-03-01", "2018-03-01"]),
  "seller_state": ["SP", "PR", "SP", "SP", "PR"],
  "comision_pct": [10.0, 12.0, 11.0, 12.5, 13.0],
}).astype({"seller_state": sellers["seller_state"].dtype})
comisiones

In [ ]:
ventas = (
  items[["order_id", "seller_id", "price"]]
  .merge(orders[["order_id", "order_purchase_timestamp"]], on="order_id", validate="m:1")
  .merge(sellers[["seller_id", "seller_state"]], on="seller_id", validate="m:1")
  .query("seller_state in ['SP', 'PR']")
  .sort_values("order_purchase_timestamp")
)

con_comision = pd.merge_asof(
  ventas,
  comisiones.sort_values("vigente_desde"),
  left_on="order_purchase_timestamp",
  right_on="vigente_desde",
  by="seller_state",
  direction="backward",
)
con_comision["comision"] = con_comision["price"] * con_comision["comision_pct"] / 100

con_comision.groupby(["seller_state", "comision_pct"], observed=True).agg(
  desde=("order_purchase_timestamp", "min"),
  hasta=("order_purchase_timestamp", "max"),
  ventas=("price", "size"),
  comision=("comision", "sum"),
).round({"comision": 2})

Cada venta tomó la comisión vigente **para su estado** en la fecha de compra. Con un `merge` normal sería imposible: las fechas casi nunca coinciden exactamente.

## 10 · `combine_first`

Rellena los faltantes de un objeto con los valores de otro, **alineando por índice**. Útil cuando tienes una fuente principal y otra de respaldo.

Ejemplo: el estado de un cliente según `customers` (principal) y según la tabla de geolocalización (respaldo).

In [ ]:
estado_principal = customers.set_index("customer_id")["customer_state"].astype("str")
estado_principal.iloc[:5] = np.nan                  # simulamos faltantes

estado_geo = (
  customers.merge(geo_zip, left_on="customer_zip_code_prefix",
                  right_on="geolocation_zip_code_prefix", how="left")
  .set_index("customer_id")["state"].astype("str")
)

completo = estado_principal.combine_first(estado_geo)
print("Faltantes:", estado_principal.isna().sum(), "->", completo.isna().sum())

---
# ✍️ Ejercicios

Variables disponibles: `orders`, `items`, `payments`, `reviews`, `customers`, `sellers`, `geo`, `geo_zip`, `products`, `haversine_km`.

### Ejercicio 1 · Contratos y deduplicación ⭐

1. `n_sin_review`: número de pedidos **sin ninguna** reseña (anti-join).
2. `review_peor`: una reseña por pedido, quedándote con la de **menor** `review_score`; en empate, la de `review_answer_timestamp` más **antigua**.
3. `orders_r`: `orders` + `review_score` de `review_peor`, con `how="left"` y `validate="1:1"`.
4. `n_cambia`: ¿en cuántos pedidos el score elegido es distinto al de la reseña **más antigua** (por `review_answer_timestamp`)?

In [ ]:
# TU CÓDIGO AQUÍ
n_sin_review = ...
review_peor = ...
orders_r = ...
n_cambia = ...

In [ ]:
# Comprobación
if pendiente(n_sin_review, review_peor, orders_r, n_cambia):
  print("⏳ Ejercicio 1 pendiente: resuélvelo en la celda de arriba")
else:
  assert n_sin_review == 1
  assert review_peor["order_id"].is_unique and len(review_peor) == reviews["order_id"].nunique()
  assert len(orders_r) == len(orders) and "review_score" in orders_r.columns
  assert orders_r["review_score"].isna().sum() == 1
  assert n_cambia == 1, n_cambia
  print("✅ Ejercicio 1 correcto")

### Ejercicio 2 · Conciliación pedido vs pago ⭐⭐

Construye `concil`, **una fila por pedido de `orders`**, con:

| columna | definición |
|---|---|
| `n_items` | número de ítems |
| `valor_items` | suma de `price + freight_value` |
| `pagado` | suma de `payment_value` |
| `diferencia` | `pagado - valor_items` |

Usa `validate` en cada `merge`. Después:

- `n_sin_items`: pedidos sin ningún ítem (`valor_items` faltante).
- `n_descuadre`: pedidos donde `abs(diferencia) > 0.01` (ignora los que tengan `NaN`).
- `cuotas_descuadre`: número **máximo** de cuotas (`payment_installments`) entre los pagos de los pedidos con descuadre. ¿Se te ocurre por qué un pedido podría pagar más que el valor de sus ítems?

In [ ]:
# TU CÓDIGO AQUÍ
concil = ...
n_sin_items = ...
n_descuadre = ...
cuotas_descuadre = ...

In [ ]:
# Comprobación
if pendiente(concil, n_sin_items, n_descuadre, cuotas_descuadre):
  print("⏳ Ejercicio 2 pendiente: resuélvelo en la celda de arriba")
else:
  assert len(concil) == len(orders) and concil["order_id"].is_unique
  assert {"n_items", "valor_items", "pagado", "diferencia"} <= set(concil.columns)
  assert np.isclose(concil["pagado"].sum(), payments["payment_value"].sum())
  assert np.isclose(concil["valor_items"].sum(), (items["price"] + items["freight_value"]).sum())
  assert n_sin_items == 1, n_sin_items
  assert n_descuadre == 1, n_descuadre
  assert cuotas_descuadre == 10, cuotas_descuadre
  print("✅ Ejercicio 2 correcto")

### Ejercicio 3 · Distancias por estado ⭐⭐

Usa `coord_vendedor` (coordenadas de vendedor con respaldo) y `geo_zip` de la sección 6, pero escribe tu propio pipeline de uniones:

1. `dist`: una fila por **ítem**, con `seller_state`, `customer_state` y `km` (distancia vendedor → cliente). Usa `validate` en cada `merge`.
2. `pct_sin_km`: % de ítems sin distancia (por coordenadas faltantes).
3. `km_por_estado`: mediana de `km` por **estado del cliente**, sólo estados con al menos 10 ítems con distancia, ordenada descendente.
4. `mismo_estado_km`: mediana de `km` para ítems donde vendedor y cliente están en el mismo estado.

In [ ]:
# TU CÓDIGO AQUÍ
dist = ...
pct_sin_km = ...
km_por_estado = ...
mismo_estado_km = ...

In [ ]:
# Comprobación
if pendiente(dist, pct_sin_km, km_por_estado, mismo_estado_km):
  print("⏳ Ejercicio 3 pendiente: resuélvelo en la celda de arriba")
else:
  assert len(dist) == len(items)
  assert {"seller_state", "customer_state", "km"} <= set(dist.columns)
  assert np.isclose(pct_sin_km, 10.119, atol=0.01), pct_sin_km
  assert km_por_estado.is_monotonic_decreasing
  assert list(km_por_estado.index.astype(str)) == ["BA", "RS", "PR", "MG", "SC", "RJ", "SP"], list(km_por_estado.index.astype(str))
  assert np.isclose(mismo_estado_km, 21.348, atol=0.1), mismo_estado_km
  print("✅ Ejercicio 3 correcto")

### Ejercicio 4 · `concat` de eventos ⭐⭐

1. `log`: tabla larga con columnas `order_id`, `evento`, `ts`, construida con **un solo `concat`** a partir de las 4 fechas de `orders`: `order_purchase_timestamp` → `"compra"`, `order_approved_at` → `"aprobado"`, `order_delivered_carrier_date` → `"enviado"`, `order_delivered_customer_date` → `"entregado"`. Sin filas con `ts` faltante. `evento` debe ser una **categoría ordenada** en ese orden.
2. `ultimo`: para cada pedido, su **último** evento alcanzado (según el orden de la categoría, no la fecha). Series indexada por `order_id`.
3. `conteo_ultimo`: `value_counts` de `ultimo` (incluyendo categorías con 0).
4. `n_incoherentes`: pedidos donde algún evento tiene `ts` **anterior** al evento previo en el orden lógico.

In [ ]:
# TU CÓDIGO AQUÍ
log = ...
ultimo = ...
conteo_ultimo = ...
n_incoherentes = ...

In [ ]:
# Comprobación
if pendiente(log, ultimo, conteo_ultimo, n_incoherentes):
  print("⏳ Ejercicio 4 pendiente: resuélvelo en la celda de arriba")
else:
  assert list(log.columns) == ["order_id", "evento", "ts"]
  assert isinstance(log["evento"].dtype, pd.CategoricalDtype) and log["evento"].cat.ordered
  assert list(log["evento"].cat.categories) == ["compra", "aprobado", "enviado", "entregado"]
  assert log["ts"].notna().all() and len(log) == orders[["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]].notna().sum().sum()
  assert len(ultimo) == len(orders)
  assert {str(k): int(v) for k, v in conteo_ultimo.items()} == {"compra": 0, "aprobado": 3, "enviado": 4, "entregado": 293}, conteo_ultimo.to_dict()
  assert n_incoherentes == 6, n_incoherentes
  print("✅ Ejercicio 4 correcto")

### Ejercicio 5 · Reto: `merge_asof` con tipo de cambio ⭐⭐⭐

Una tabla **sintética** (inventada para el ejercicio, no son tasas reales) de tipo de cambio BRL → MXN publicada cada lunes desde marzo de 2017:

In [ ]:
fechas_fx = pd.date_range("2017-03-06", "2018-09-03", freq="W-MON")
fx = pd.DataFrame({
  "fecha_fx": fechas_fx,
  "brl_mxn": np.round(5.6 + 0.4 * np.sin(np.arange(len(fechas_fx)) / 8), 4),
})
# Simulamos que en diciembre de 2017 no se publicó la tasa
fx = fx[~fx["fecha_fx"].between("2017-12-01", "2017-12-31")].reset_index(drop=True)
fx.head()

1. `pagos_mxn`: cada fila de `payments` con su fecha de compra (de `orders`), la **última tasa publicada en o antes** de esa fecha, con tolerancia de **10 días** (si no hay tasa en ese rango, `NaN`), y `valor_mxn = payment_value * brl_mxn`.
2. `n_sin_tasa`: pagos sin tasa.
3. `mxn_por_mes`: suma de `valor_mxn` por mes de compra (`Period` mensual), excluyendo pagos sin tasa.
4. **Pregunta de razonamiento:** ¿por qué hay pagos sin tasa en dos periodos distintos? Escribe tu respuesta en un comentario.

In [ ]:
# TU CÓDIGO AQUÍ
pagos_mxn = ...
n_sin_tasa = ...
mxn_por_mes = ...

In [ ]:
# Comprobación
if pendiente(pagos_mxn, n_sin_tasa, mxn_por_mes):
  print("⏳ Ejercicio 5 pendiente: resuélvelo en la celda de arriba")
else:
  assert len(pagos_mxn) == len(payments)
  assert {"brl_mxn", "valor_mxn"} <= set(pagos_mxn.columns)
  assert n_sin_tasa == 25, n_sin_tasa
  assert isinstance(mxn_por_mes.index, pd.PeriodIndex)
  assert np.isclose(mxn_por_mes.sum(), 226469.702, atol=0.5), mxn_por_mes.sum()
  print("✅ Ejercicio 5 correcto")

---
# ✅ Soluciones

### Solución 1

In [ ]:
n_sin_review = (~orders["order_id"].isin(reviews["order_id"])).sum()

review_peor = (
  reviews.sort_values(["review_score", "review_answer_timestamp"])
         .drop_duplicates("order_id", keep="first")
)
orders_r = orders.merge(review_peor[["order_id", "review_score"]], on="order_id",
                        how="left", validate="1:1")

primera_review = (reviews.sort_values("review_answer_timestamp")
                          .drop_duplicates("order_id", keep="first"))
comparacion = review_peor.merge(primera_review, on="order_id", suffixes=("_peor", "_primera"),
                                validate="1:1")
n_cambia = (comparacion["review_score_peor"] != comparacion["review_score_primera"]).sum()
n_sin_review, n_cambia

*Nota:* `sort_values` con dos llaves + `drop_duplicates(keep="first")` es el patrón estándar para "quedarse con una fila por llave según un criterio". Los `suffixes` hacen legible la comparación.

### Solución 2

In [ ]:
it_ped = (items.assign(valor=items["price"] + items["freight_value"])
               .groupby("order_id", as_index=False)
               .agg(n_items=("order_item_id", "size"), valor_items=("valor", "sum")))

pg_ped = payments.groupby("order_id", as_index=False).agg(pagado=("payment_value", "sum"))

concil = (
  orders[["order_id", "order_status"]]
  .merge(it_ped, on="order_id", how="left", validate="1:1")
  .merge(pg_ped, on="order_id", how="left", validate="1:1")
  .assign(diferencia=lambda d: d["pagado"] - d["valor_items"])
)

n_sin_items = concil["valor_items"].isna().sum()

descuadre = concil[concil["diferencia"].abs() > 0.01]
n_descuadre = len(descuadre)
cuotas_descuadre = payments.loc[payments["order_id"].isin(descuadre["order_id"]), "payment_installments"].max()
n_sin_items, n_descuadre, cuotas_descuadre, descuadre

*Nota:* `abs(NaN) > 0.01` es `False`, así que el pedido sin ítems queda fuera automáticamente. En esta muestra los pagos cuadran casi perfecto; el único descuadre es un pago a **10 cuotas**, consistente con **intereses** por pagar a plazos. Una conciliación así es típica en análisis contables.

### Solución 3

In [ ]:
coords = geo_zip.set_index("geolocation_zip_code_prefix")[["lat", "lng"]]

dist = (
  items[["order_id", "order_item_id", "seller_id"]]
  .merge(sellers[["seller_id", "seller_state"]], on="seller_id", validate="m:1")
  .merge(orders[["order_id", "customer_id"]], on="order_id", validate="m:1")
  .merge(customers[["customer_id", "customer_zip_code_prefix", "customer_state"]], on="customer_id", validate="m:1")
  .join(coord_vendedor.add_prefix("v_"), on="seller_id")
  .join(coords.add_prefix("c_"), on="customer_zip_code_prefix")
  .assign(km=lambda d: haversine_km(d["v_lat"], d["v_lng"], d["c_lat"], d["c_lng"]))
)

pct_sin_km = dist["km"].isna().mean() * 100

km_por_estado = (
  dist.dropna(subset=["km"])
      .groupby("customer_state", observed=True)["km"]
      .agg(["size", "median"])
      .query("size >= 10")["median"]
      .sort_values(ascending=False)
)

mismo_estado_km = dist.loc[
  dist["seller_state"].astype(str) == dist["customer_state"].astype(str), "km"
].median()
round(pct_sin_km, 2), km_por_estado.round(1), round(mismo_estado_km, 1)

*Nota:* `km` es `NaN` cuando falta cualquiera de las dos coordenadas (NumPy propaga `NaN`); aquí sólo faltan las de algunos clientes. Ojo: con el respaldo por capital, la distancia "mismo estado" es aproximada (vendedor ubicado en la capital). Comparamos los estados como `str` porque son dos `category` con listas de categorías distintas.

### Solución 4

In [ ]:
COLS = {
  "order_purchase_timestamp": "compra",
  "order_approved_at": "aprobado",
  "order_delivered_carrier_date": "enviado",
  "order_delivered_customer_date": "entregado",
}
evento_dtype = pd.CategoricalDtype(list(COLS.values()), ordered=True)

log = (
  pd.concat(
    [orders[["order_id", col]].rename(columns={col: "ts"}).assign(evento=nombre)
     for col, nombre in COLS.items()],
    ignore_index=True,
  )
  .dropna(subset=["ts"])
  .astype({"evento": evento_dtype})
  [["order_id", "evento", "ts"]]
)

ultimo = log.groupby("order_id")["evento"].max()
conteo_ultimo = ultimo.value_counts().sort_index()

log_ord = log.sort_values(["order_id", "evento"])
retroceso = log_ord.groupby("order_id")["ts"].diff() < pd.Timedelta(0)
n_incoherentes = log_ord.loc[retroceso, "order_id"].nunique()
conteo_ultimo, n_incoherentes

*Notas:* el `max()` de una categoría **ordenada** respeta su orden lógico. Para la incoherencia ordenamos por el orden lógico del evento y comprobamos con `diff()` que las fechas no retrocedan. Encontrar pedidos "aprobados antes de comprados" o "entregados antes de enviados" es un hallazgo típico de calidad de datos.

### Solución 5

In [ ]:
pagos_fecha = (
  payments.merge(orders[["order_id", "order_purchase_timestamp"]], on="order_id", validate="m:1")
          .sort_values("order_purchase_timestamp")
)

pagos_mxn = pd.merge_asof(
  pagos_fecha,
  fx.sort_values("fecha_fx"),
  left_on="order_purchase_timestamp",
  right_on="fecha_fx",
  direction="backward",
  tolerance=pd.Timedelta(days=10),
).assign(valor_mxn=lambda d: d["payment_value"] * d["brl_mxn"])

n_sin_tasa = pagos_mxn["brl_mxn"].isna().sum()

mxn_por_mes = (
  pagos_mxn.dropna(subset=["valor_mxn"])
           .groupby(pagos_mxn["order_purchase_timestamp"].dt.to_period("M"))["valor_mxn"]
           .sum()
)

# Respuesta a la pregunta 4:
# 1) Pagos antes del 2017-03-06: no existe ninguna tasa anterior (backward no encuentra nada).
# 2) Pagos de ~2017-12-08 a 2017-12-31: la última tasa disponible es la del 2017-11-27,
#    a más de 10 días, así que la tolerancia los deja sin tasa.
n_sin_tasa, pagos_mxn.loc[pagos_mxn["brl_mxn"].isna(), "order_purchase_timestamp"].dt.to_period("M").value_counts().sort_index()

---
## Resumen del módulo

- Antes de unir, revisa la **cardinalidad** de la llave en ambos lados.
- Usa `validate` en cada `merge`: convierte supuestos en errores visibles.
- **Agrega antes de unir** las tablas "muchos" para evitar la explosión m:m.
- Llaves: mismo **dtype** y mismo **formato** (los ceros a la izquierda importan).
- Semi/anti-join con `isin`; con llaves compuestas, `indicator=True`.
- `join` para unir contra tablas indexadas; `map` para traer una sola columna.
- `concat` apila; nunca dentro de un bucle.
- `merge_asof` para "lo vigente en esa fecha", con `by`, `direction` y `tolerance`.
- `combine_first` para completar una fuente con otra.

**Siguiente:** Módulo 06 · Series de tiempo: `resample`, ventanas móviles, zonas horarias y periodos.